In [8]:
# ============================================================
# Recompute IPW on concluded accepted loans (subset)
# ============================================================

!pip install xgboost==1.7.6 --quiet

import os
import gc
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

# ---------------- CONFIG ----------------
DATA_PATH = "accepted_stage1_preprocessed.csv"     # <-- 1.3M concluded loans
ARTIFACT_DIR = "approval_artifacts"

OUTPUT_FILE = "accepted_with_ipw_final_subset.csv"

CHUNK = 200_000
PROP_MIN = 0.01
PROP_MAX = 0.99
WINSOR_PCT = 99.0

# 🔒 FIXED global acceptance rate (DO NOT recompute)
OVERALL_ACCEPT_RATE = 0.07553166433301935

BASE_YEAR = 2007

# ---------------- LOAD ARTIFACTS ----------------
bst = xgb.Booster()
bst.load_model(os.path.join(ARTIFACT_DIR, "xgb_model_v2.json"))

iso = joblib.load(os.path.join(ARTIFACT_DIR, "isotonic_v2.joblib"))
cat_maps = joblib.load(os.path.join(ARTIFACT_DIR, "cat_maps_v2.joblib"))
freq_maps = joblib.load(os.path.join(ARTIFACT_DIR, "freq_maps_train_only.joblib"))
feature_cols = joblib.load(os.path.join(ARTIFACT_DIR, "feature_cols_v2.joblib"))

zip3_freq = freq_maps["zip3_freq"]
state_freq = freq_maps["state_freq"]
cat_cols = list(cat_maps.keys())

print("Artifacts loaded successfully.")

# ---------------- FEATURE ENGINEERING ----------------
def apply_mappings_and_features(df_in):
    df = df_in.copy()

    # categorical codes
    for c in cat_cols:
        df[c + "_code"] = (
            df[c].astype(str)
            .fillna("nan")
            .map(lambda v: cat_maps[c].get(v, -1))
            .astype("int32")
        )

    # frequency encodings
    df["zip3_freq"] = (
        df["zip3"].astype(str).fillna("nan").map(lambda v: zip3_freq.get(v, 0.0)).astype("float32")
    )
    df["state_freq"] = (
        df["state_cat"].astype(str).fillna("nan").map(lambda v: state_freq.get(v, 0.0)).astype("float32")
    )

    # time features
    m = pd.to_numeric(df["months_since_start"], errors="coerce").fillna(-999).astype("int32")
    df["application_year"] = (BASE_YEAR + (m // 12)).astype("int32")
    df["application_month"] = ((m % 12) + 1).astype("int32")

    df["month_sin"] = np.sin(2 * np.pi * df["application_month"] / 12).astype("float32")
    df["month_cos"] = np.cos(2 * np.pi * df["application_month"] / 12).astype("float32")

    # numeric cleanup
    for c in ["emp_length_clean", "amount_requested", "dti_clean", "months_since_start"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-999).astype("float32")

    # interactions
    df["dti_x_time"] = (df["dti_clean"] * df["months_since_start"]).astype("float32")
    df["amount_x_time"] = (df["amount_requested"] * df["months_since_start"]).astype("float32")

    return df

# ---------------- STREAMING IPW COMPUTATION ----------------
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

rows_written = 0
ipw_values = []

print("Starting IPW recomputation on concluded subset...")

for i, chunk in enumerate(pd.read_csv(DATA_PATH, chunksize=CHUNK, low_memory=False)):
    print(f"Processing chunk {i}")

    chunk = apply_mappings_and_features(chunk)

    X = chunk[feature_cols].astype("float32")
    dmat = xgb.DMatrix(X)

    p_raw = bst.predict(dmat)
    p_cal = iso.predict(p_raw)
    p = np.clip(p_cal, PROP_MIN, PROP_MAX)

    # IPW (all rows here are accepted & concluded)
    ipw = OVERALL_ACCEPT_RATE / p

    ipw_values.append(ipw)

    out = pd.DataFrame({
        "propensity_accept_calibrated": p,
        "ipw_raw": ipw
    })

    out.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=(rows_written == 0),
        index=False
    )

    rows_written += len(out)
    del chunk, X, dmat, p_raw, p_cal, p, ipw, out
    gc.collect()

print("Raw IPW computation complete. Rows:", rows_written)

# ---------------- POSTPROCESS: WINSORIZE + NORMALIZE ----------------
df_ipw = pd.read_csv(OUTPUT_FILE)

cap = np.percentile(df_ipw["ipw_raw"], WINSOR_PCT)
df_ipw["ipw_clipped"] = np.minimum(df_ipw["ipw_raw"], cap)

# Normalize: sum(weights) = N
df_ipw["ipw_final"] = df_ipw["ipw_clipped"] * (len(df_ipw) / df_ipw["ipw_clipped"].sum())

# ESS
w = df_ipw["ipw_final"].values
ess = (w.sum() ** 2) / (np.square(w).sum())

print("IPW percentiles:", np.percentile(df_ipw["ipw_final"], [50, 75, 90, 95, 99]))
print("ESS:", ess, "N:", len(df_ipw))

df_ipw.to_csv("accepted_with_ipw_final_subset.csv", index=False)
print("Saved accepted_with_ipw_final_subset.csv")


Artifacts loaded successfully.
Starting IPW recomputation on concluded subset...
Processing chunk 0
Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Raw IPW computation complete. Rows: 1369177
IPW percentiles: [ 0.27168241  0.41982646  1.87327552  4.34115363 16.92157781]
ESS: 185599.53283830525 N: 1369177
Saved accepted_with_ipw_final_subset.csv


In [4]:
import pandas as pd
import os

file_path = "accepted_with_ipw_final.csv"  # replace with your file path

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("File loaded successfully. Showing first 5 rows:")
    print(df.head())
else:
    print(f"File not found: {file_path}")


File loaded successfully. Showing first 5 rows:
   row_id  propensity_accept  propensity_accept_calibrated  ipw_weight  \
0       0           0.990694                      0.875074    0.086315   
1       1           0.987579                      0.826553    0.091381   
2       2           0.938044                      0.530118    0.142481   
3       3           0.693201                      0.121719    0.620541   
4       4           0.994267                      0.910028    0.082999   

   approval_status  ipw_normalized  
0                1        0.165663  
1                1        0.175388  
2                1        0.273462  
3                1        1.190998  
4                1        0.159300  


In [10]:
import pandas as pd
import os

file_path = "accepted_with_ipw_final_subset.csv"  # replace with your file path

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("File loaded successfully. Showing first 5 rows:")
    print(df.head())
else:
    print(f"File not found: {file_path}")


File loaded successfully. Showing first 5 rows:
   propensity_accept_calibrated   ipw_raw  ipw_clipped  ipw_final
0                      0.875074  0.086315     0.086315   0.193373
1                      0.826553  0.091381     0.091381   0.204725
2                      0.530118  0.142481     0.142481   0.319204
3                      0.910028  0.082999     0.082999   0.185946
4                      0.936443  0.080658     0.080658   0.180701
